# 03 · Agreement, adjudication → *your* gold set

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egumasa/lda2-final-template/blob/main/notebooks/03_annotate.ipynb)

The part no model can do for you, and the part the Q&A will ask about.

```
  01_build_pool_<track>  →  02_sample  →  02b_add_samples  →▶ 03_annotate  →  04_develop  →  05_test  →  06_report
```

| | |
|---|---|
| **Reads** | `data/gold/<track>_<group>_sample.json` and the sheet (both from 02) |
| **Writes** | `data/gold/<track>_<group>_gold.json`, and its `_dev.json` / `_test.json` split |

---

**Come here when both coders have finished.** Notebook 02 drew the sample and made the sheet; this one turns two people's labels into one gold set.

The published labels are somebody else's judgment. You re-annotated the sample blind; now you find out how far apart the two of you were, and argue out the rows you disagreed on.

What comes out is *your* gold set — and the disagreements tell you which label boundaries are genuinely fuzzy. That is what lets you say, later, whether a model's miss is the **model's** fault or the **scheme's**. Nothing else in the project can tell you that, and notebook 06 asks you for it directly.

It ends by drawing one more line: which of your annotated items you are allowed to *look at* while you write prompts, and which are held back for the number you report. One sheet, one adjudication, then a split — it costs no extra coding.

## Setup — run this first

This cell mounts your Google Drive and finds your group's shared folder, `lda2-final-template`. Everything the project produces — the pool, the gold set, your prompts, the outputs — is an ordinary file in there, which is what makes it survive the runtime resetting *and* lets the rest of your group see it.

**One member sets the folder up once:**

1. That member runs the `git clone` line this cell prints if the folder is missing, which puts it in their own Drive.
2. They share it with the group (right-click ▸ *Share*), with edit access.
3. Everyone else opens *Shared with me*, right-clicks the folder, and chooses **Add shortcut to Drive** ▸ *My Drive*.

Keep that shortcut's name exactly `lda2-final-template`. It is what makes the same path work for all of you — if Drive renames it to `lda2-final-template (1)`, this cell will not find it.

From then on, open notebooks from the folder itself (*File ▸ Open notebook ▸ Drive*) rather than from the GitHub badge, so you are working on your group's copy and not a fresh one.

**Looking inside a helper.** The functions this cell imports are defined in `scripts/`. Two ways to read one, both the same ones you used on Day 2:

- `help(save_json)` prints its first line — what to pass in and what comes back — and the description of each argument. Typing `save_json(` and pressing **Shift+Tab** shows the same thing in a pop-up.
- To read the code itself, open `scripts/pipeline.py` from the **Files** panel on the left. Colab lists the functions in that file down the side, so you can click straight to the one you want.

In [ ]:
# ------------------------------------------------------------------
# SETUP — run me first. You are not expected to read it.
# ------------------------------------------------------------------
# This cell is plumbing, and it is the only cell in the project that is.
# It finds your group's shared folder in Google Drive, because everything
# this project keeps goes in there: a Colab runtime is wiped when it resets,
# and nobody else in your group can see inside it. Then it makes the
# project's own code importable. Run it and move on; nothing below asks you
# to have understood it.

FOLDER = "lda2-final-template"     # the shared folder, in every member's Drive

import os, sys

PROJECT = ".."                              # running locally: it is just above us

try:
    from google.colab import drive           # only exists inside Colab
except ImportError:
    pass
else:
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/" + FOLDER
    if not os.path.isdir(PROJECT):
        raise RuntimeError(
            "Could not find " + PROJECT + "\n\n"
            "Setting the folder up for your group? Run this in a new cell:\n"
            "  !git clone https://github.com/egumasa/lda2-final-template.git "
            + PROJECT + "\n"
            "then share the folder with the rest of your group.\n\n"
            "Someone else already did? Open Drive, find the folder under "
            "'Shared with me', right-click it, and choose 'Add shortcut to "
            "Drive'. Keep the name exactly " + FOLDER + ".")
    # Work inside the project folder, where the notebooks live.
    os.makedirs(PROJECT + "/notebooks", exist_ok=True)
    os.chdir(PROJECT + "/notebooks")

# scripts/ and config.py, by their real paths - so they are found from wherever
# this notebook happens to be working.
sys.path.append(PROJECT)
sys.path.append(PROJECT + "/scripts")

# Re-read config.yaml every time this cell runs. Without the reload, Python
# hands back the settings it read the FIRST time, and editing config.yaml
# would appear to do nothing until you restarted the runtime.
import importlib
import config
importlib.reload(config)

# Named one by one rather than with `import *`, so that every name a cell
# below uses can be traced back to the file it came from — config.yaml for
# these, scripts/ for the rest.
from config import (TRACK, GROUP, RUN, SEED, N_PER_CLASS, DEV, CODERS,
                    MEMBERS, LABELS_ORDER, TEMPERATURE, MODEL, ROOT, OUT_DIR,
                    POOL_PATH, DEMO_POOL_PATH, SAMPLE_PATH, GOLD_PATH,
                    SAMPLE_BEFORE_TOPUP_PATH, DEV_PATH, TEST_PATH,
                    DISAGREED_PATH, ADJUDICATED_PATH, PRED_PATH, ROUNDS_PATH,
                    NOTES_PATH, TESTLOG_PATH, PROMPT_FILE, SHEET_PATH,
                    describe)

# The Google Sheets round trip is plumbing, so it is imported, and so is the
# dev/test split: stratifying and rounding are bookkeeping, not judgment.
from pipeline import (load_gold, label_set, save_json, split_dev_test,
                      plot_confusion_matrix)
from annotate import (remembered_sheet, load_coder_sheets, fleiss_kappa,
                      adjudicated_rows)

# The sheet's column headings, by the names the code below uses for them.
from _study import COL_ID, COL_TEXT, COL_FINAL

# `column`, `percent_agreement`, `to_canonical` and `compare_to_published` are
# NOT imported. Each one decides something you have to defend — which rows count,
# what a valid label is, what counts as the same item — so they are cells you can
# read and change instead. Run those cells before the steps that use them.

# The scoring itself comes from scikit-learn, by its own names. You checked your
# hand-built versions of these against it on Day 2 S6; these are those functions.
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# pandas, for the table the function you write in step 3 hands back.
import pandas as pd

# `disagreements` is NOT imported. Step 3 asks you to write it, because the rule
# inside it — what counts as a disagreement — is a decision about your scheme
# rather than a fact about your data.

describe()                  # what this notebook is working on


> **Everything above comes from `config.yaml`** — one small file at the top of the repo, which you edit once as a group, and the only file in the plumbing you touch. That is deliberate: the seed that drew your sample has to be the seed you report, and five copies of a number in five notebooks is five chances for them to disagree. Your settings are also the filenames — `track: cars50`, `group: kimura`, `run: v1` means this notebook reads and writes `cars50_kimura_v1_...`. If the line it just printed is not your track, your group and your seed, fix `config.yaml` and re-run this cell.

## First — your sample, back from the file

Notebook 02 saved it, and this is the moment that was for. Days have passed, the runtime that drew the sample is long gone, and the person running this cell may not be the person who ran 02.

It matters that this is a **load and not a redraw**: the sheet your coders filled in was built from these exact forty items, and adjudication puts their labels back onto them one by one. `to_canonical` also uses this list to restore what the sheet does not carry — on `cars50` and `raamove`, the passage each sentence came from.

In [ ]:
sampled = load_gold(SAMPLE_PATH)
LABELS = label_set(sampled)

print(len(sampled), "items ·", LABELS)

# These still carry the PUBLISHED label. Ignore it for now — you compare
# against it in step 5, once your own labels are settled.


## Then — the annotation sheet, found again

Now we look up the sheet notebook 02 created, from the small file it wrote the link to. That file is why the link is not lost: whoever runs this notebook need not be the person who ran notebook 02, and need not still have that cell's output on screen.

If this prints *none saved yet*, notebook 02's sheet step has not been run — or was run by somebody whose copy of the folder is not this one.

In [ ]:
SHEET_ID = remembered_sheet(SHEET_PATH)

# Working on a sheet someone made before this file existed? Paste its URL (or
# just the long id from it) here instead:
# SHEET_ID = ""

print("sheet:", SHEET_ID or "-- none saved yet: run notebook 02 --")


**`column`** turns one coder's tab into a plain list, in row order. Note what it does with a cell that coder left blank: it keeps an empty string rather than dropping the row, so two calls on the same rows stay the same length and stay lined up item by item. Dropping would silently shift one coder's labels against the other's.

In [ ]:
def column(rows: list[dict[str, str]], name: str) -> list[str]:
    """Pull one coder's labels out of the merged sheet rows, in row order.

    The sheet arrives as one dict per item, with a column per coder. Almost everything
    you want to measure needs two plain lists lined up item by item instead, so this is
    usually the first line of an agreement cell.

    Rows where that coder left the cell blank come back as an empty string rather than
    being dropped, so two calls on the same rows stay the same length and stay aligned.

    Args:
        rows: the merged rows, one per item, as load_coder_sheets returns them.
        name: the column to read, e.g. "CoderA".

    Returns:
        That column's labels, one per row, in row order.

    Example:
        >>> a_labels = column(rows, "CoderA")
    """
    labels = []
    for row in rows:
        labels.append(str(row.get(name, "")).strip())
    return labels

**`percent_agreement`** counts how often the two matched. Read the `if` in the middle: the denominator is **rows both coders labelled**, not every row in your sample. A pair who have each finished half the sheet can show a high agreement over very few items.

This matters for the cell you write in step 2. `cohen_kappa_score` does **not** apply that filter — hand it the same two lists and it counts a blank as a label like any other. Two numbers, side by side in your report, over different sets of rows.

In [ ]:
def percent_agreement(labels_a: list[str], labels_b: list[str]) -> float:
    """How often two coders chose the same label, counting only rows both labelled.

    This is the number to report next to a kappa, never instead of one: it counts the
    agreement you would get by chance as though you had earned it. Two coders using one
    label for nine items in ten agree 90% of the time without reading anything.

    It is written out rather than taken from sklearn's accuracy_score, which computes
    the same fraction. "Accuracy" names one of the two lists as correct, and when the
    two are coders neither of them is.

    Args:
        labels_a: the first coder's labels.
        labels_b: the second coder's labels, same items, same order.

    Returns:
        The proportion of doubly-labelled rows where the two match, 0.0 to 1.0.

    Raises:
        ValueError: when the two lists are different lengths, which would compare
            the wrong sentences with each other.

    Example:
        >>> percent_agreement(a_labels, b_labels)
    """
    if len(labels_a) != len(labels_b):
        raise ValueError(
            "The two lists of labels are different lengths: the first has "
            + str(len(labels_a)) + " and the second has " + str(len(labels_b))
            + ". They have to line up item by item, or the comparison pairs the wrong "
            "sentences together.\nMost often one coder left rows blank at the bottom "
            "of their tab. Fill them in and run the cell again.")

    both_labelled = 0
    matched = 0
    for index in range(len(labels_a)):
        if labels_a[index] and labels_b[index]:      # skip rows nobody finished
            both_labelled = both_labelled + 1
            if labels_a[index] == labels_b[index]:
                matched = matched + 1

    if both_labelled == 0:
        print("No rows where both coders have labelled. Nothing to compare yet.")
        return 0.0

    share = matched / both_labelled
    print(both_labelled, "doubly-annotated ·", matched, "matched · agreement",
          format(share, ".1%"))
    return share

## Step 1 — Line the coders up, and get the first number

Each coder has their **own tab**, so the first thing to do is line them up side by side. `load_coder_sheets` reads one tab per name you give it and joins them by item id into a single table — one column per coder, plus `Final`.

> **Who annotated?** `CODERS` comes from `config.yaml`, so notebook 06 finds the same tabs. If a third coder joined, duplicate an **empty** tab in the sheet (right-click ▸ *Duplicate*), rename it `CoderC`, and add it there.

`column(rows, name)` pulls one coder's labels out as a plain list, in row order — two lists lined up item by item, which is what every statistic takes.

Then percent agreement: how often the two of them chose the same label. It is the one number every design owes, whatever else you report, and it is the only one this cell computes. **Step 2 is where you add the rest**, because which ones you owe depends on your design rather than on your data.

Run it once **every** coder's tab is filled in — rows that not everyone labelled are left out of the comparison. If two coders appear to have given every item the same label, somebody duplicated a tab that had already been filled in; that agreement is a copy rather than a measurement.

Now we read the tabs, line them up, and measure raw agreement.

In [ ]:
# ══ STEP 1 · Line the coders up ═══════════════════════════════════════════
# Reads one tab per coder, lines them up by item id, and measures how often
# they chose the same label.
# Creates: rows, a_labels, b_labels

rows = load_coder_sheets(SHEET_ID, CODERS)   # one read per tab, merged by ID

a_labels = column(rows, CODERS[0])
b_labels = column(rows, CODERS[1])

percent_agreement(a_labels, b_labels)


## Step 2 — The rest of the agreement your design owes

Percent agreement counts lucky agreement as if you had earned it. Two coders labelling at random on a two-label scheme agree half the time; the same number on an eight-label scheme means something else entirely. So it never stands alone.

**What you owe is not a free choice, and it is settled before you run anything** — by how many coders you have, and by whether `PLAN.md` §3 says your labels are a scale:

| Your design | Report |
|---|---|
| two coders, labels with no order | percent agreement **and** Cohen's κ |
| two coders, labels on a scale | those two, **and** the weighted κ |
| three or more coders | percent agreement **and** Fleiss' κ, plus Cohen's κ per pair |

Read that off your own design. Choosing the statistic after seeing which one flatters you is the one thing that would make the number meaningless — which is exactly why the rule above depends on nothing you are about to find out.

Here is everything available. Nothing here is new: you met the last three on Day 2 S6, under these names, when you checked your own precision, recall, F1 and κ against scikit-learn's.

| Call | What it gives you |
|---|---|
| `cohen_kappa_score(a, b)` | agreement corrected for chance, two coders |
| `cohen_kappa_score(a, b, weights="quadratic")` | the same, counting a near miss as a smaller error |
| `fleiss_kappa([a, b, c])` | one number for three or more coders |
| `confusion_matrix(a, b, labels=LABELS)` | which label **pairs** you disagree about |
| `plot_confusion_matrix(matrix, LABELS, title)` | that matrix, drawn |

**The confusion matrix is not optional**, whichever numbers you report. The κ says how far apart you were; only the off-diagonal cells say *which pair of labels* you disagree about, and that pair is what you go back to the sheet to argue about. Looking at it changes what you do next, so there is nothing to protect yourself from.

**Write the numbers down as they print** — they belong in your report's methodology section, and they do not survive a runtime reset. A κ around .8 is strong; around .4 means the scheme, not the annotators, is doing something wrong. Either is a reportable finding. A low κ you can explain beats a high one you cannot.

Now we add the statistics your design owes, and draw the matrix. `a_labels` and `b_labels` are ready from step 1.

Stuck? `from answers import answer`, then `answer("agreement")` — after you have tried.

In [ ]:
# ══ STEP 2 · The rest of the agreement your design owes ═══════════════════
# Adds the chance-corrected statistic your design calls for, and draws the
# coder-vs-coder confusion matrix.
# Creates: matrix

# ✏️ your code here


**✍️ For your report and the Q&A** — this goes in the write-up, not in a cell below.

> Our coders agreed on ___% of items, with a ___ κ of ___.
>
> We report ___ as well as percent agreement because our labels ___.
>
> The pair we disagreed about most often was ___ and ___, which suggests our scheme ___.

The second sentence comes off the table above, and the reason is what is graded — not which statistic you ran.

The third is the one the Q&A goes to. It comes off the matrix, and it is a claim about your **scheme** rather than about your coders — two labels that keep swapping usually means the boundary between them is not written down clearly enough. It is also what step 3 acts on.

A low κ you can explain beats a high one you cannot.

## Step 3 — What counts as a disagreement?

Now you need the list of rows to argue about. **This one you write**, and it is the only function in the project that you do — because the rule inside it is a decision about your scheme rather than a fact about your data, and there is no way to hand it over without answering it for you.

The obvious rule: a row is a disagreement when your coders did not all choose the same label. For most schemes that is the right one.

It is not the only defensible one. **If your labels sit on a scale** — A1 < A2 < … < C2, Low < Mid < High — you might decide that neighbouring labels are two people reading the same sentence much the same way, and that only a gap of two or more is worth an argument. That version hands back a shorter list and sends you to the sheet with less to settle. Which you chose, and why, is a sentence in your report; not knowing which you used is the only wrong answer.

You have written this shape before. It is a loop over `rows`, keeping the ones where the labels differ:

```python
def disagreements(rows, coders):
    out = []
    for row in rows:
        ...        # pull each coder's label out of this row
        ...        # keep the row if they are not all the same
    return pd.DataFrame(out, columns=list(rows[0]))
```

The cell below it calls yours as `disagreements(rows, coders=CODERS)` — by keyword, so that it works whether your second parameter is named `coders` or you pasted the reference version, whose first two parameters are the two column names.

`column(rows, name)` is not what you want inside here — that reads a whole column down the sheet, and you are working across one row at a time. `row.get(name, "")` is the piece you need, and `str(...).strip()` around it drops the stray spaces a spreadsheet loves to add.

**Leave out the rows nobody finished.** A blank cell is a coder who has not got there yet, not two people disagreeing, and counting it as a disagreement puts an item on your adjudication list that has nothing to adjudicate.

Tried it? `from answers import answer`, then `answer("disagreements")`.

In [ ]:
# ══ STEP 3 · Write the rule ═══════════════════════════════════════════════
# Defines the function that decides which rows go on your adjudication list.
# Nothing runs until the cell below calls it.
# Creates: disagreements

# ✏️ your code here


### Now run it, and keep the result

This cell calls the function you just wrote and saves what comes back.

The table comes back in notebook 06, where the rows your coders argued about are what you check the model's errors against — and saving it here means 05 does not have to sign back in to the sheet and derive the same table a second time. It also means that step still works after the sheet has been deleted, or its owner has left.

The last line is just the name `disagreed`, with no `print`. In a notebook the value of a cell's last line is displayed automatically, and for a table that reads far better than `print` would. Add a line after it and the table stops appearing, which is the one thing to watch out for.

In [ ]:
# `coders=` by name, not by position: the reference version in answers.py
# takes the two column names first, so a pasted copy of it would read CODERS
# as one coder's name if this were positional.
disagreed = disagreements(rows, coders=CODERS)
save_json(disagreed.to_dict("records"), DISAGREED_PATH,
          what="rows your coders disagreed on")
disagreed

**✍️ For your report and the Q&A** — this goes in the write-up, not in a cell below.

> We counted a row as a disagreement when ___.
>
> That put ___ of ___ items on our adjudication list.
>
> We chose that rule rather than ___ because ___.

Both rules are defensible and they hand you different lists, so the sentence that matters is the third one. If your labels are not on a scale there was only one sensible rule, and saying so is a complete answer.

## Step 4 — Adjudicate

The rows left on your list do not go away by rewriting the guidelines. You decide them. This is the Day 2 S5 step F, on your own data.

Go back to the sheet and fill in `Final` for **every** row:

- Where you agreed, `Final` is that label.
- Where you did not, talk it out and decide. If you cannot agree, the scheme is underspecified — write down *why* in `Note` and pick one. That note is worth more to your report than the label is.

### Before you settle them: is a second round worth it?

Look again at the confusion matrix from step 2. **If one pair of labels accounts for most of your disagreements**, the boundary between those two is not written down clearly enough, and that is fixable: revise what your guidelines say about that pair, duplicate the coder tabs into a fresh round, re-annotate the affected rows there, and re-run steps 1–3. Agreement should move, and you can report by how much.

**If the disagreements are scattered across many pairs**, a second round re-measures the same fuzziness and κ will barely move. Adjudicate and go on.

Never overwrite a round — keep each one as its own tab, so the change is something you can show rather than assert. Either way, write down which of the two you found and what you did about it. That is the answer to *what did your QC pass change?*, which is published in advance as a Q&A question.

Then re-read the sheet and canonicalise it. `to_canonical` reports blanks and invalid labels rather than silently dropping them; fix them in the sheet and re-run until it says **0 blank, 0 invalid**. A blank row is an item that has gone missing from your study without telling you.

It passes `source=sampled`: gold is rebuilt from the **sheet**, which carries only the id, the text and your label, so anything else the item had — on `cars50` and `raamove`, its passage — is put back from `sampled` by id. On the other tracks that argument does nothing.

### The code that builds your gold set — read it, then run it

`to_canonical` is the function that decides what your gold set **is**, so what it leaves out matters as much as what it keeps. Read the three piles it sorts every row into:

- A **blank** `Final` is counted and skipped. That row does not reach your gold set.
- A label **not in your scheme** is reported, not repaired. `.strip()` is the only cleaning it does, so `b1` and `B11` are invalid rather than read as `B1` — it will not guess what you meant.
- A row whose **ID cell** is not a number is dropped and named.

So `len(gold)` can be smaller than the number of items you sampled, and nothing raises an error when it is. That is why it prints all three counts: **run it until it says 0 blank and 0 invalid**, or report the shortfall as a limitation.

In [ ]:
def to_canonical(rows: list[dict[str, str]],
                 labels: list[str],
                 column: str = COL_FINAL,
                 source: list[dict[str, str]] | None = None
                 ) -> list[dict[str, str]]:
    """Turn annotation rows into canonical gold: [{"id", "text", "label"}, ...].

    Blank rows are skipped; labels outside `labels` are reported, not silently kept.

    Args:
        rows: the rows read back by load_coder_sheets or load_annotation_sheet.
        labels: the labels your scheme allows. Anything else is reported as invalid.
        column: which column holds the adjudicated label.
        source: the items the sheet was BUILT from - your sampled items. Pass it on a
            track that carries context: gold is rebuilt from the sheet, which holds
            only the id, the text and your label, so anything else the item was
            carrying would be dropped here and notebook 04 would never see it. The
            extra fields are copied from `source` by id rather than read back out of
            the sheet, because the sheet's Context column is a marked-up display copy
            a coder may have edited.

    Returns:
        The usable rows as gold items, each {"id", "text", "label"} plus whatever
        `source` was carrying.

    Example:
        >>> gold = to_canonical(rows, LABELS_ORDER, source=sampled)
    """
    ### Step 0: look up what each sampled item was carrying, if we were given them ###
    extras_by_id = {}
    if source is not None:
        for item in source:
            extras = {}
            for key in item:
                if key not in ("id", "text", "label"):
                    extras[key] = item[key]
            extras_by_id[item["id"]] = extras

    ### Step 1: sort every row into one of three piles ###
    gold = []          # usable rows
    blank = 0          # not labelled yet
    invalid = []       # typos, wrong case, labels that are not in the scheme
    bad_ids = []       # rows whose ID cell is not a number
    for row in rows:
        label = str(row.get(column, "")).strip()   # .strip() drops stray spaces
        if not label:
            blank = blank + 1                      # nobody has filled this row in yet
        elif label not in labels:
            invalid.append((row.get(COL_ID), label))   # e.g. "b1" or "B11"
        else:
            # The ID column should hold the number the sheet was created with. If
            # someone has typed over it, say which row rather than crash.
            try:
                item_id = int(row[COL_ID])
            except (KeyError, TypeError, ValueError):
                bad_ids.append(row.get(COL_ID))
                continue
            gold_item = {
                "id": item_id,
                "text": str(row[COL_TEXT]),
                "label": label,
            }
            gold_item.update(extras_by_id.get(item_id, {}))   # Put back whatever the sheet could not carry.
            gold.append(gold_item)

    ### Step 2: report every count, so nothing is dropped silently ###
    print(len(gold), "usable ·", blank, "still blank ·", len(invalid), "invalid")
    if invalid:
        print("  fix these in the sheet, then re-run:", invalid[:10])   # first 10
        print("  allowed labels:", ", ".join(labels))
    if bad_ids:
        print("  these rows have a non-numeric ID cell (did something get typed over "
              "it?):", bad_ids[:10])

    ### Step 3: if we were given the sampled items, check they actually matched ###
    # A silent miss here is nasty: gold would come out looking fine, just without the
    # context, and only notebook 04 would notice - by prompting with an empty passage.
    if source is not None:
        unmatched = 0
        for item in gold:
            if item["id"] not in extras_by_id:
                unmatched = unmatched + 1
        if unmatched:
            print("  WARNING:", unmatched, "row(s) had no match in `source` by id, so "
                  "they carry no context. Is `source` the same sampled items this sheet "
                  "was created from?")
    return gold

Now we re-read the sheet, with `Final` filled in, and turn it into your gold set. `rows` from step 1 was fetched before you adjudicated, so it is fetched again here.

In [ ]:
# ══ STEP 4 · Adjudicate, then canonicalise ════════════════════════════════
# Re-reads the sheet now that Final is filled in, and turns it into your gold
# set — reporting any row that is blank or has a label it does not recognise.
# Creates: gold

# Re-read: `rows` from step 1 was fetched before you filled in Final.
rows = load_coder_sheets(SHEET_ID, CODERS)

gold = to_canonical(rows, LABELS, source=sampled)   # re-attaches what the sheet drops


### Now keep the argument, not just the answer

`to_canonical` took your `Final` labels into `gold` and dropped everything else — including the `Note` column. The label is what the model gets scored against; the note is the only record of **what you decided and on what grounds**, and it is what your report's methodology section asks you for.

So this saves the disagreed rows with both. It is one file being written now, rather than a sentence you try to reconstruct from memory in a week — and by then the sheet may have been deleted, or its owner may have left the group.

It tells you how many rows still have no `Final`, and how many have no note. A row with a label and no note is a decision nobody can check.

In [ ]:
adjudicated = adjudicated_rows(rows, coders=CODERS)
save_json(adjudicated, ADJUDICATED_PATH,
          what="rows you adjudicated, with the reason")
pd.DataFrame(adjudicated)

**✍️ For your report and the Q&A** — this goes in the write-up, not in a cell below.

> Our adjudication settled ___ rows, ___ of which we recorded a reason for.
>
> Most of them were the ___ / ___ boundary, which our scheme did not settle because ___.
>
> We did / did not run a second annotation round, because ___.

The second sentence is the finding: it names a boundary in your scheme, and the confusion matrix in step 2 is the evidence for it. Notebook 06 asks the same question of the model's errors, and the interesting result is when the two land on the same pair.

The third comes off the decision rule above — say which of the two patterns you saw, not just what you did.

## Step 5 — Where do you differ from the published labels?

Now — and only now, with your own labels settled — look at what the corpus said. `compare_to_published` matches by text and shows you every row where your group landed somewhere else.

**Disagreement here is not an error.** You annotated forty items carefully against a scheme you had thought about; the original annotators worked at scale under different guidelines. Where you differ, one of three things is true, and saying which is exactly the analytical work this project is for:

1. **Your scheme drifted** from theirs — you read a category boundary differently. Say where.
2. **The item is genuinely ambiguous** — it would split any pair of annotators.
3. **One of you is wrong.** It happens, in both directions.

This table belongs in your report's methodology section, and it is the one that most often produces a sentence worth saying out loud in the Q&A. Pick two or three rows and write down which of the three cases above they are — now, while you still remember the argument you had about them.

The comparison runs against `sampled`, not `pool`: sampling renumbered the ids, so `pool` would line your item 7 up against a completely different sentence. This is the Day 2 S5 step F call.

### The code that compares you to the published labels — read it, then run it

This function prints the percentage that goes into your report's methodology section, so read how it gets there. Three decisions are inside it:

- **What counts as the same item.** It matches on the **text**, because sampling renumbered the ids — an id match would line your item 7 up against a different sentence. Ids are a fallback for when the text has been edited, and it says so when that happens.
- **What happens to an item it cannot match at all.** Look at the `else: continue`. That item leaves the comparison entirely — it is not counted as an agreement or a difference, and the denominator shrinks without a warning.
- **What counts as differing.** Exact string equality. No case folding.

The percentage it prints is therefore over the items it **could** match, not over your whole gold set. If those two numbers are not the same, say which one you are quoting.

In [ ]:
def compare_to_published(gold: list[dict[str, str]],
                         published: list[dict[str, str]]) -> pd.DataFrame | None:
    """How often does YOUR final label match the published one, item by item?

    Items are matched by their TEXT, not their id. Sampling renumbers the ids from 1,
    so an id-based match would line YOUR item 7 up against POOL item 7 - two unrelated
    sentences - and report a meaningless number without ever failing. (Ids are still
    used as a fallback, for the case where the texts have been edited.)

    Args:
        gold: your own gold items, from to_canonical.
        published: the published items, from load_gold.

    Returns:
        A table of the items where you and the published label differ, or None when
        nothing could be matched.

    Example:
        >>> compare_to_published(my_gold, published)
    """
    ### Step 1: look up the published label for every text ###
    label_by_text = {}
    label_by_id = {}
    for item in published:
        label_by_text[str(item["text"])] = item["label"]
        label_by_id[item["id"]] = item["label"]

    ### Step 2: pair each of your items with its published label ###
    matched_rows = []
    matched_by_id_only = 0
    for item in gold:
        text = str(item["text"])
        if text in label_by_text:
            theirs = label_by_text[text]
        elif item["id"] in label_by_id:
            theirs = label_by_id[item["id"]]
            matched_by_id_only = matched_by_id_only + 1
        else:
            continue
        matched_rows.append({
            "id": item["id"],
            "yours": item["label"],
            "published": theirs,
            "text": item["text"],
        })

    if len(matched_rows) == 0:
        print("None of your items could be matched to the published set. Are you "
              "comparing against the same data you sampled from?")
        return None
    if matched_by_id_only > 0:
        print("  note:", matched_by_id_only, "item(s) matched by id because the text "
              "no longer matches exactly.")

    ### Step 3: count the matches, then show only the rows where you differ ###
    agree = 0
    differences = []
    for row in matched_rows:
        if row["yours"] == row["published"]:
            agree = agree + 1
        else:
            differences.append(row)
    print(agree, "/", len(matched_rows), "match the published label",
          "(" + format(agree / len(matched_rows), ".1%") + ")")
    return pd.DataFrame(differences)

Now we run it against the items you sampled — `sampled`, not `pool`, because those are the same forty items your coders saw.

In [ ]:
# ══ STEP 5 · Compare against the published labels ═════════════════════════
# Shows every row where your group's label and the corpus's label differ.
# Creates: differences

differences = compare_to_published(gold, sampled)   # sampled, not pool: same 40 items
differences


**✍️ For your report and the Q&A** — this goes in the write-up, not in a cell below.

> We matched ___ of our ___ gold items to the published set, and agreed with the published label on ___% of them.
>
> Where we differed, ___ of the rows were our scheme reading a boundary differently, ___ were genuinely ambiguous, and ___ were one of us being wrong.
>
> The clearest example is item ___, where we said ___ and they said ___, because ___.

The second sentence is the analytical work this step exists for, and the third is what the Q&A goes to. Pick the rows now, while you still remember the argument you had about them.

## Save it — this is the handoff

This file is the single most valuable thing your group makes all week — hours of judgment, and the only thing in the project that could not have been produced by a script. Every number in notebooks 04, 05 and 06 is measured against it, and it goes in your submission bundle.

**Next:** open `04_develop.ipynb`. It starts by loading `data/gold/<track>_<group>_gold.json`.

In [ ]:
save_json(gold, GOLD_PATH, what="gold items")

# It is git-ignored — it is your work, not part of the template. If you cloned
# into Google Drive it is already saved across sessions; if not, download it.


## Step 6 — Draw the line: dev and test

In notebook 04 you will change your prompt because of what you saw it get wrong. That is the work. But a score measured on the same items you kept adjusting against stops being a measure of how good your prompt is, and becomes a measure of **how long you kept adjusting**. It only ever goes up.

So the line gets drawn now, before anything has been run against these items:

| | what it is for |
|---|---|
| **dev** | the items you may look at. Iterate here, as many rounds as you like. |
| **test** | opened once, in `05_test.ipynb`. Whatever it says is what you report. |

Both halves came out of the same sheet and the same adjudication, so the split costs you no extra annotation. What it costs is items you are allowed to learn from — which is why the ratio is a real decision and `PLAN.md` §6 asks you to defend the one you made. A bigger dev gives steadier feedback while you iterate and leaves a smaller test, so the number you finally report bounces more; a smaller dev means prompt decisions made on very few items, which is how you tune to noise and then watch the gain evaporate.

This also replaces the old advice to keep `n_per_class` at 2 while iterating. **dev is the fast set now** — a dozen or so items is about a minute per round, and your sample stays at full size throughout.

The split is stratified by label, so both halves keep every label wherever the data allows. Where it does not — a label with a single surviving item — that item goes to **test**, and the function says so. That asymmetry is deliberate: a label missing from test drops out of the macro average without announcing itself, while a label missing from dev only costs you feedback.

### What `split_dev_test` does with your `dev:` setting

It is imported rather than printed here, because what is inside it is bookkeeping — grouping by label, rounding a fraction to a whole number of items, and being careful about a label with only one item left. None of that is a decision you make; the ratio is, and that is in `config.yaml`.

Three things it does that are worth knowing, because they show up in your numbers:

- **A rare class goes to test, not dev.** A label with a single surviving item cannot be on both sides. Missing from test, it drops out of your macro average without announcing itself; missing from dev, it only costs you feedback. The second is the cheaper mistake, so that is the one it makes.
- **The rounding is written out** rather than left to `round()`, which in Python rounds 0.5 down and 1.5 up — not something you want to explain in the Q&A.
- **The ids are not renumbered.** Notebook 06 asks which of the model's errors are also the rows your coders argued about, and that join runs on these ids.

`help(split_dev_test)` prints what to pass it. To read the code itself, run `split_dev_test??` — it prints the source of any function, imported or not.

Now we draw the line: which of your gold items you are allowed to look at while iterating, and which you are not. Nothing here existed in Days 1–3 — no set there was worth holding back. `split_dev_test` is the function you defined and read just above.

**Run this once, and before you open notebook 04.** Splitting again after you have iterated on dev means the held-out items have already been seen — by you, if not by the model.

How big dev is comes from `dev:` in `config.yaml`, and how you wrote the number says what you meant. A balanced draw (`sample_pool`) suits a whole number, `dev: 3` — three items per label. An uneven one (`sample_random`) suits a decimal, `dev: 0.35` — a third of each label, because a fixed 3 per class would eat a small class whole.

Nothing is saved yet — read the counts it prints first.

In [ ]:
# ══ STEP 6 · Split dev / test ═════════════════════════════════════════════
# Splits your gold set in two, keeping every label on both sides wherever the
# data allows, and prints how many items each half got.
# Creates: dev, test

# DEV comes from config.yaml: a whole number is items per label, a decimal
# is a proportion of each label.
#
# Drew your sample with sample_by_document? Add by_document=True inside the
# brackets below, so that no passage has some of its sentences in dev and
# the rest in test.
dev, test = split_dev_test(gold, DEV, seed=SEED)


### Now save both halves

Read the per-label counts the split just printed **before** you run this. A label that lands in dev but not in test cannot appear in the score you report, and this is the last easy moment to change `dev` in `config.yaml` and draw the line again.

Once you are happy, save. Notebook 04 opens `dev`; notebook 05 opens `test`, once.

In [ ]:
save_json(dev,  DEV_PATH,  what="dev items")
save_json(test, TEST_PATH, what="test items")

**✍️ For your report and the Q&A** — this goes in the write-up, not in a cell below.

> We split ___ gold items into ___ dev and ___ test.
>
> We set `dev:` to ___ because ___.
>
> Every label is present on both sides except ___.

There is no right ratio at this size, only one you can defend, so the second sentence is the whole answer. The third is the limitation a reader needs in order to read your per-label scores — a label that is only in test was never something you could iterate against.

---

## 🛑 The `PLAN.md` gate

Notebook 04 starts calling the model. **Do not open it until your `PLAN.md` has been read and signed off.** It takes two minutes and it is not busywork: a mismatched label set or an unstated sampling seed costs an hour to unpick *after* you have burned quota on it.

Check, out loud, that these three agree: the label set in `PLAN.md`, the labels `label_set` actually returned above, and the labels your prompt file names. And that `PLAN.md` records **which sampling strategy you chose, and why**, and **§6: which split spec you set, the sizes it produced, and why that ratio**.